# 02 — Baseline: Seasonal-Naive (Store Sales)

**Goal of this notebook:** establish the *first number to beat*. Before any learned model, we measure a deliberately simple forecast — the **seasonal-naive** baseline — on the shared 16-day validation set. Every later stage (deterministic, features, hybrid) is judged by how much it improves on this score (Constitution II).

The baseline rule is the simplest thing that respects weekly seasonality: **predict each day with the observed sales from the same weekday of the prior week.** Over the 16-day horizon this just repeats each series' last complete training week.

Run it top-to-bottom from a fresh kernel.

## 0. Setup

Put the repo root on the import path so the reusable `src/` package is importable whether the kernel started in `notebooks/` or at the repo root. All loading, splitting, scoring, and the model itself live in `src/` — the notebook stays thin and the logic is reused across stages.

In [1]:
import sys
from pathlib import Path

# The kernel's working directory is `notebooks/`, but our reusable code lives in the
# project's `src/` package one level up. Put the repo root on the import path so
# `import src.data` works whether the kernel started in the repo root or in notebooks/.
here = Path.cwd()
repo_root = here if (here / "src").exists() else here.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.data as data
import src.validation as validation
import src.models as models

print("repo root:", repo_root)

repo root: D:\AIO\M01\Conquer\TimeSeries


## 1. Load and reindex the data (gap-free)

Load `train.csv` and reindex every `(store_nbr, family)` series onto a complete daily calendar. Closed days (absent from the raw file) become `sales = 0`, so the per-series index is gap-free — a precondition for both the weekday alignment below and every later lag/seasonality feature. We re-assert the gap-free guarantee here so the baseline is built on trustworthy data.

In [2]:
raw = data.load_train()
df = data.reindex_series_gapfree(raw)
validation.assert_gapfree(df)

print(f"raw rows:       {len(raw):,}")
print(f"gap-free rows:  {len(df):,}")
print(f"date range:     {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"series:         {df[['store_nbr', 'family']].drop_duplicates().shape[0]:,}")

raw rows:       3,000,888
gap-free rows:  3,008,016
date range:     2013-01-01 -> 2017-08-15
series:         1,782


## 2. Build the validation split

Carve off the **last 16 days** as a time-based validation set (2017-07-31 → 2017-08-15). It mirrors the real forecast horizon and comes strictly *after* the training data, so the local RMSLE behaves like the leaderboard and no future day leaks into training. Every stage scores on this exact same window.

In [3]:
train, val = validation.train_validation_split(df)

print(f"train: {train['date'].min().date()} -> {train['date'].max().date()}  ({len(train):,} rows)")
print(f"validation: {val['date'].min().date()} -> {val['date'].max().date()}  ({len(val):,} rows)")
print(f"validation days: {val['date'].nunique()}")

train: 2013-01-01 -> 2017-07-30  (2,979,504 rows)
validation: 2017-07-31 -> 2017-08-15  (28,512 rows)
validation days: 16


## 3. Fit and predict the seasonal-naive baseline

`seasonal_naive_predict` reads only the **training** half (never the validation set), so it is leak-free by construction. For each series it maps the last training week's `weekday → sales`, then assigns each of the 16 validation days the value from its matching weekday. The result is a tidy `[store_nbr, family, date, sales_pred]` frame — one row per series × validation day (1,782 × 16 = 28,512).

In [4]:
preds = models.seasonal_naive_predict(train, validation.HORIZON_DAYS)

print("prediction rows:", f"{len(preds):,}")
print("NaN predictions:", int(preds['sales_pred'].isna().sum()))
preds.head()

prediction rows: 28,512
NaN predictions: 0


,store_nbr,family,date,sales_pred
0,1,AUTOMOTIVE,2017-07-31,4.0
1,1,AUTOMOTIVE,2017-08-01,10.0
2,1,AUTOMOTIVE,2017-08-02,2.0
3,1,AUTOMOTIVE,2017-08-03,5.0
4,1,AUTOMOTIVE,2017-08-04,7.0


## 4. Score on the validation set (RMSLE, log space)

Merge the predictions onto the validation actuals by `(store_nbr, family, date)`, then compute **RMSLE** — the competition metric. `rmsle` takes *raw* sales and applies `log1p` internally (so we pass raw actuals and raw predictions, not pre-logged values) and clips predictions to `>= 0`. An exact-match merge with no missing rows confirms predictions and actuals are perfectly aligned.

In [5]:
scored = val.merge(preds, on=['store_nbr', 'family', 'date'], how='inner')
assert len(scored) == len(val), 'prediction/validation misalignment'
assert scored[['sales', 'sales_pred']].isna().sum().sum() == 0, 'missing values in scored frame'

baseline_rmsle = validation.rmsle(scored['sales'], scored['sales_pred'])
print(f"scored rows: {len(scored):,}")
print(f"baseline validation RMSLE: {baseline_rmsle:.5f}")

scored rows: 28,512
baseline validation RMSLE: 0.61704


## 5. Record the baseline in the iteration log

Append this score to `iteration_log.md` via `log_iteration` — the single, append-only record of every technique's validation RMSLE and its delta versus the best so far. This is the **first real entry** (it replaces the *baseline pending* placeholder) and the bar every later stage must beat (SC-002).

> Note: this cell writes to `iteration_log.md`. Re-running it appends another row — expected for an append-only log; the log is reviewed for completeness in Stage 8.

In [6]:
delta = validation.log_iteration(
    "baseline: seasonal-naive (weekly)",
    baseline_rmsle,
    notes="same-weekday-prior-week; repeats last training week over 16-day horizon",
)
print(f"logged baseline RMSLE={baseline_rmsle:.5f}  delta={delta}")

logged baseline RMSLE=0.61704  delta=+0.10713 (worse)


## Takeaway

The seasonal-naive baseline gives a validation RMSLE of **~0.617**. It captures weekly seasonality but nothing else — no trend, no holidays, no promotions, no oil. That's exactly why it's the bar to beat.

**Next (Stage 2 — `03_deterministic.ipynb`):** add a trend term and Fourier seasonality with `LinearRegression`, targeting at least a 10% relative RMSLE reduction versus this baseline (SC-003).